In [ ]:
from collections import defaultdict
from pathlib import Path
from time import perf_counter

import pydicom


In [ ]:
dataset_path = Path("../data/raw/dicom_test_studies_old")

In [ ]:
files = [
    path
    for path in dataset_path.rglob("*")
    if path.is_file()
]

print(f"Archivos encontrados: {len(files)}")
for path in files:
    print(path)


for study_dir in sorted(dataset_path.iterdir()):
    if study_dir.is_dir():
        count = sum(1 for path in study_dir.rglob("*") if path.is_file())
        print(f"{study_dir.name}: {count} archivos")

Archivos encontrados: 154
..\data\raw\dicom_test_studies\study_01\instance-0001.dcm
..\data\raw\dicom_test_studies\study_01\instance-0002.dcm
..\data\raw\dicom_test_studies\study_01\instance-0003.dcm
..\data\raw\dicom_test_studies\study_01\instance-0004.dcm
..\data\raw\dicom_test_studies\study_01\instance-0005.dcm
..\data\raw\dicom_test_studies\study_01\instance-0006.dcm
..\data\raw\dicom_test_studies\study_01\instance-0007.dcm
..\data\raw\dicom_test_studies\study_01\instance-0008.dcm
..\data\raw\dicom_test_studies\study_01\instance-0009.dcm
..\data\raw\dicom_test_studies\study_01\instance-0010.dcm
..\data\raw\dicom_test_studies\study_01\instance-0011.dcm
..\data\raw\dicom_test_studies\study_01\instance-0012.dcm
..\data\raw\dicom_test_studies\study_01\instance-0013.dcm
..\data\raw\dicom_test_studies\study_01\instance-0014.dcm
..\data\raw\dicom_test_studies\study_01\instance-0015.dcm
..\data\raw\dicom_test_studies\study_01\instance-0016.dcm
..\data\raw\dicom_test_studies\study_01\instan

In [27]:
processed_count = 0
error_count = 0

metadata = []

start = perf_counter()

for path in files:
    try:
        ds = pydicom.dcmread(
            path,
            stop_before_pixels=True,
            specific_tags=[
                "StudyInstanceUID",
                "SeriesInstanceUID",
                "SOPInstanceUID",
                "Modality",
                "SeriesDescription",
                "InstanceNumber",
            ],
        )

        metadata.append({
            "path": path,
            "study_uid": getattr(ds, "StudyInstanceUID", None),
            "series_uid": getattr(ds, "SeriesInstanceUID", None),
            "sop_uid": getattr(ds, "SOPInstanceUID", None),
            "modality": getattr(ds, "Modality", None),
            "series_description": getattr(
                ds,
                "SeriesDescription",
                None,
            ),
            "instance_number": getattr(
                ds,
                "InstanceNumber",
                None,
            ),
        })
        
        processed_count += 1

    except (
        pydicom.errors.InvalidDicomError,
        OSError,
        EOFError,
    ):
        error_count += 1

elapsed = perf_counter() - start

print(f"Archivos encontrados: {len(files)}")
print(f"Archivos procesados: {processed_count}")
print(f"Archivos con errores: {error_count}")
print(f"DICOM válidos: {len(metadata)}")
print(f"Tiempo: {elapsed:.4f} segundos")

Archivos encontrados: 154
Archivos procesados: 154
Archivos con errores: 0
DICOM válidos: 154
Tiempo: 1.5404 segundos


In [13]:
for item in metadata[:3]:
    print(item)

print(metadata[0]["study_uid"])
print(metadata[0]["series_uid"])
print(metadata[0]["instance_number"])

{'path': WindowsPath('../data/raw/dicom_test_studies/study_01/instance-0001.dcm'), 'study_uid': '1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597', 'series_uid': '1.3.6.1.4.1.14519.5.2.1.311251873679098488391157318475172879927', 'sop_uid': '1.3.6.1.4.1.14519.5.2.1.211656320229857259150742995635142122704', 'modality': 'MR', 'series_description': '_T1_Brain      STEALTH 2MM', 'instance_number': '20'}
{'path': WindowsPath('../data/raw/dicom_test_studies/study_01/instance-0002.dcm'), 'study_uid': '1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597', 'series_uid': '1.3.6.1.4.1.14519.5.2.1.311251873679098488391157318475172879927', 'sop_uid': '1.3.6.1.4.1.14519.5.2.1.142974816986116801185309998545028886409', 'modality': 'MR', 'series_description': '_T1_Brain      STEALTH 2MM', 'instance_number': '69'}
{'path': WindowsPath('../data/raw/dicom_test_studies/study_01/instance-0003.dcm'), 'study_uid': '1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597', 'serie

In [23]:
studies = defaultdict(list)

for item in metadata:
    studies[item["study_uid"]].append(item)

In [26]:
print(f"Studies encontrados: {len(studies)}")

for study_uid, instances in studies.items():
    print(
        f"Study {study_uid}: "
        f"{len(instances)} archivos"
    )

Studies encontrados: 2
Study 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597: 90 archivos
Study 1.3.6.1.4.1.14519.5.2.1.4720853842805410700493332290090736697: 64 archivos


In [24]:
study_series = {}

for study_uid, instances in studies.items():
    
    series_groups = defaultdict(list)
    
    for instance in instances:
        series_groups[
            instance["series_uid"]
        ].append(instance)
    
    study_series[study_uid] = series_groups

In [25]:
for study_uid, series_groups in study_series.items():

    print(f"\nStudy: {study_uid}")
    print(f"Cantidad de series: {len(series_groups)}")

    for series_uid, instances in series_groups.items():

        print(
            f"  Series: {series_uid} "
            f"→ {len(instances)} archivos"
        )


Study: 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
Cantidad de series: 1
  Series: 1.3.6.1.4.1.14519.5.2.1.311251873679098488391157318475172879927 → 90 archivos

Study: 1.3.6.1.4.1.14519.5.2.1.4720853842805410700493332290090736697
Cantidad de series: 1
  Series: 1.3.6.1.4.1.14519.5.2.1.52263584488698073404576488693793099986 → 64 archivos


In [18]:
for item in metadata[:10]:
    print(
        item["path"].parent.name,
        "→",
        item["study_uid"]
    )

uids_by_folder = defaultdict(set)

for item in metadata:
    folder = item["path"].parent.name
    uids_by_folder[folder].add(item["study_uid"])

for folder, uids in uids_by_folder.items():
    print(f"{folder}:")
    for uid in uids:
        print(f"  {uid}")


study_01 → 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_01 → 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_01 → 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_01 → 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_01 → 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_01 → 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_01 → 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_01 → 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_01 → 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_01 → 1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_01:
  1.3.6.1.4.1.14519.5.2.1.11784217633835209751588758740160819597
study_02:
  1.3.6.1.4.1.14519.5.2.1.4720853842805410700493332290090736697


In [28]:
big_path = Path("../data/raw/dicom_test_studies")

files = [
    path
    for path in big_path.rglob("*")
    if path.is_file()
]

print(f"Archivos encontrados: {len(files)}")

Archivos encontrados: 43719


In [34]:
studies = defaultdict(lambda: defaultdict(int))

processed_count = 0
error_count = 0
errors = []

start = perf_counter()

for big_path in files:
    try:
        ds = pydicom.dcmread(
            big_path,
            stop_before_pixels=True,
            specific_tags=[
                "StudyInstanceUID",
                "SeriesInstanceUID",
            ],
        )

        study_uid = getattr(ds, "StudyInstanceUID", None)
        series_uid = getattr(ds, "SeriesInstanceUID", None)

        if study_uid is None or series_uid is None:
            error_count += 1
            errors.append({
            "file": str(path),
            "reason": "Falta StudyInstanceUID o SeriesInstanceUID",
            })
            continue

        studies[study_uid][series_uid] += 1
        processed_count += 1

    except (
        pydicom.errors.InvalidDicomError,
        OSError,
        EOFError,
    ) as e:
        error_count += 1
        
        errors.append({
            "file": str(path),
            "reason": str(e),
            "type": type(e).__name__
        })

elapsed = perf_counter() - start

In [35]:
print(f"Archivos encontrados: {len(files)}")
print(f"Archivos procesados: {processed_count}")
print(f"Archivos con errores: {error_count}")
print(f"Studies encontrados: {len(studies)}")

series_count = sum(
    len(series_groups)
    for series_groups in studies.values()
)

print(f"Series encontradas: {series_count}")
print(f"Tiempo: {elapsed:.4f} segundos")

Archivos encontrados: 43719
Archivos procesados: 43718
Archivos con errores: 1
Studies encontrados: 31
Series encontradas: 343
Tiempo: 539.8736 segundos


In [36]:
print("\n=== ERRORES ===")

for error in errors:
    print(f"\nArchivo: {error['file']}")
    print(f"Tipo: {error.get('type', 'N/A')}")
    print(f"Motivo: {error['reason']}")


=== ERRORES ===

Archivo: ..\data\raw\dicom_test_studies\study_02\instance-0064.dcm
Tipo: InvalidDicomError
Motivo: File is missing DICOM File Meta Information header or the 'DICM' prefix is missing from the header. Use force=True to force reading.


In [38]:
bad_file = Path(
    r"..\data\raw\dicom_test_studies_old\study_02\instance-0064.dcm"
)

print(f"Tamaño: {bad_file.stat().st_size} bytes")

with open(bad_file, "rb") as f:
    header = f.read(132)

print(header)

Tamaño: 109362 bytes
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00DICM'


In [40]:
bad_file = Path(
    r"..\data\raw\dicom_test_studies_old\study_02\instance-0064.dcm"
)

try:
    ds = pydicom.dcmread(
        bad_file,
        force=True
    )

    print("Lectura con force=True exitosa")
    print(ds)

except Exception as e:
    print(f"Error: {type(e).__name__}")
    print(f"Motivo: {e}")

Lectura con force=True exitosa
Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 204
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: MR Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.6.1.4.1.14519.5.2.1.214791505187822567877081114652741302553
(0002,0010) Transfer Syntax UID                 UI: Implicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.3.6.1.4.1.22213.1.143
(0002,0013) Implementation Version Name         SH: '0.5'
(0002,0016) Source Application Entity Title     AE: 'POSDA'
-------------------------------------------------
(0008,0005) Specific Character Set              CS: 'ISO_IR 100'
(0008,0008) Image Type                          CS: ['DERIVED', 'SECONDARY', 'OTHER']
(0008,0012) Instance Creation Date              DA: '20091116'
(0008,0013) Instance Creation Time              TM: '135901.093000'
(0008,0016

In [31]:
for i, (study_uid, series_groups) in enumerate(studies.items()):
    print(
        f"Study {study_uid}: "
        f"{len(series_groups)} series"
    )

    if i >= 9:
        break

Study 1.3.6.1.4.1.9590.100.1.2.163777218220154630427022086743807596529: 12 series
Study 1.3.6.1.4.1.9590.100.1.2.313794056600413584631410478281141595485: 11 series
Study 1.3.6.1.4.1.9590.100.1.2.110639780812854909329128228141749292508: 11 series
Study 1.3.6.1.4.1.9590.100.1.2.194791812319135126127745958361462082440: 11 series
Study 1.3.6.1.4.1.9590.100.1.2.421488318334260161125684200712705281371: 10 series
Study 1.3.6.1.4.1.9590.100.1.2.329737130703596962422753362762909478493: 12 series
Study 1.3.6.1.4.1.9590.100.1.2.367393939729628169231456911793734075291: 12 series
Study 1.3.6.1.4.1.9590.100.1.2.106353566313551520931524579520941457759: 12 series
Study 1.3.6.1.4.1.9590.100.1.2.322893404526752461926806366740989455405: 12 series
Study 1.3.6.1.4.1.9590.100.1.2.255317365309300363526157372292299839547: 12 series


In [32]:
series_per_study = [
    len(series_groups)
    for series_groups in studies.values()
]

print(
    f"Menor cantidad de series en un estudio: "
    f"{min(series_per_study)}"
)

print(
    f"Mayor cantidad de series en un estudio: "
    f"{max(series_per_study)}"
)

print(
    f"Promedio de series por estudio: "
    f"{sum(series_per_study) / len(series_per_study):.2f}"
)

Menor cantidad de series en un estudio: 4
Mayor cantidad de series en un estudio: 12
Promedio de series por estudio: 11.06


In [33]:
for study_uid, series_groups in studies.items():
    file_count = sum(series_groups.values())

    print(
        f"Study {study_uid}: "
        f"{len(series_groups)} series, "
        f"{file_count} archivos"
    )

Study 1.3.6.1.4.1.9590.100.1.2.163777218220154630427022086743807596529: 12 series, 2172 archivos
Study 1.3.6.1.4.1.9590.100.1.2.313794056600413584631410478281141595485: 11 series, 1330 archivos
Study 1.3.6.1.4.1.9590.100.1.2.110639780812854909329128228141749292508: 11 series, 1001 archivos
Study 1.3.6.1.4.1.9590.100.1.2.194791812319135126127745958361462082440: 11 series, 1300 archivos
Study 1.3.6.1.4.1.9590.100.1.2.421488318334260161125684200712705281371: 10 series, 1810 archivos
Study 1.3.6.1.4.1.9590.100.1.2.329737130703596962422753362762909478493: 12 series, 1452 archivos
Study 1.3.6.1.4.1.9590.100.1.2.367393939729628169231456911793734075291: 12 series, 1452 archivos
Study 1.3.6.1.4.1.9590.100.1.2.106353566313551520931524579520941457759: 12 series, 1452 archivos
Study 1.3.6.1.4.1.9590.100.1.2.322893404526752461926806366740989455405: 12 series, 1061 archivos
Study 1.3.6.1.4.1.9590.100.1.2.255317365309300363526157372292299839547: 12 series, 2001 archivos
Study 1.3.6.1.4.1.9590.100.1.2